# Making the Podcaster Crew Production-Ready

In the previous notebook we built a 2-agent podcast research pipeline. It works, but production workflows need typed outputs, validation, and a third agent to produce the final script.

| Step | The Problem | The Fix | CrewAI API |
|------|------------|---------|------------|
| 1 | Agent output is raw text — parsing it is fragile | **Structured Output** | `output_pydantic=Model` |
| 2 | Bad or vague output passes through unchecked | **Code Guardrail** | `guardrail=validate_fn` |
| 3 | Writing validation functions for everything is tedious | **No-Code Guardrail** | `guardrail="plain English"` |
| 4 | We have research and a report but no script | **Scriptwriter Agent** | Third agent with context chain |
| 5 | All three agents in one run | **Full Pipeline** | `Crew(agents=[...], tasks=[...])` |


In [1]:
# !pip install -q "crewai[tools]" python-dotenv pydantic

import os
from datetime import datetime
from typing import Any, Tuple

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai.tasks.task_output import TaskOutput
from crewai_tools import EXASearchTool
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
search_tool = EXASearchTool()


/Users/ishandutta/miniconda3/envs/agents/lib/python3.10/site-packages/crewai_tools/tools/exa_tools/exa_search_tool.py:40: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  api_key: Optional[str] = Field(


We start with the same researcher from Notebook 1.


In [2]:
researcher = Agent(
    role="AI Podcast Researcher",
    goal="Find recent releases, articles, topics, and progress about {topic}",
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find the most relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=True,
    max_iter=5,
    max_rpm=12,
    max_execution_time=300,
    respect_context_window=True,
)


---

## 1. Structured Output

### The Problem: Raw Text Is Fragile

In Notebook 1, the researcher returned a wall of markdown. If downstream code needs the episode angle or a count of findings, it has to parse free-form text. That breaks constantly.

### The Fix: `output_pydantic`

Define a Pydantic model and CrewAI forces the agent to return data that matches the schema.

> **Prompt Engineering — Structured Output Specification:** Each `Field(description=...)` tells the agent exactly what to put in that field — the same technique as specifying output format directly in a prompt. The schema is the constraint.


In [3]:
# Prompt Engineering: Structured Output Specification
class ResearchItem(BaseModel):
    title: str = Field(description="Short title for the finding")
    summary: str = Field(description="What happened or why it matters")
    listener_angle: str = Field(description="Why podcast listeners would care")
    source: str = Field(description="Source name or URL when available")


class ResearchReport(BaseModel):
    topic: str = Field(description="Podcast topic being researched")
    episode_angle: str = Field(description="Main angle for the episode")
    items: list[ResearchItem] = Field(description="Research findings for the episode")
    host_questions: list[str] = Field(description="Questions hosts can ask during the episode")


In [4]:
structured_research_task = Task(
    description=(
        "Conduct podcast research about {topic}. Find recent releases, articles, topics, "
        "and progress. Use the current date context: {current_month} {current_year}."
    ),
    expected_output="A structured research report with an episode angle, findings, and host questions.",
    output_pydantic=ResearchReport,
    agent=researcher,
)

structured_crew = Crew(
    agents=[researcher],
    tasks=[structured_research_task],
    process=Process.sequential,
    verbose=True,
)

inputs = {
    "topic": "AI LLMs",
    "current_month": datetime.now().strftime("%B"),
    "current_year": str(datetime.now().year),
}

structured_result = structured_crew.kickoff(inputs=inputs)


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: a5e5daaf-2148-4803-9300-9c49a72b7024                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Task: Conduct podcast research about AI LLMs. Find recent releases, articles, topics, and progress. Use the    │
│  current date context: May 2026.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Thought: Thought: I need to find the most recent releases, articles, and topics around AI LLMs as of May 2026  │
│  to create a comprehensive podcast research report with summary, episode angle, news items, and host            │
│  questions.                                                                                                     │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"latest developments AI large language models releases progress\",                        │
│  \"start_published_date\": \"2026-03-01\", \"end_published_date\": \"2026-05-15\"}"                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {                                                                                                              │
│    "topic": "AI LLMs: The 2026 Spring Surge",                                                                   │
│    "episode_angle": "Spring 2026 has brought a wave of innovation in Large Language Models (LLMs), with new     │
│  releases pushing boundaries in performance, accessibility, specialization, and governance. We explore          │
│  OpenAI’s GPT-5.5 and its cyber-focused sibling, Baidu’s ERNIE 5.1, Google’s Gemma 4, and the open-source       │
│  momentum from MiniMax—plus the implications of rapid, ongoing LLM progress.",                                  │
│    "items": [                                                                                                   │
│      {                                                                                                          │
│        "title": "OpenAI’s GPT-5.5: A Leap Toward the 'AI Super App'",                                           │
│        "summary": "OpenAI’s GPT-5.5, unveiled in April 2026, has been hailed as introducing 'a new class of     │
│  intelligence.' This LLM vastly improves reasoning, multi-modal abilities, and integration of specialized       │
│  capabilities (notably through GPT-5.5-Cyber for trusted cyber operations). OpenAI claims its model             │
│  outperforms previous versions on numerous benchmarks and is shaping a future where LLMs underpin               │
│  multi-functional AI 'super apps.' However, the API’s doubled price has sparked debate on access and            │
│  democratization.",                                                                                             │
│        "listener_angle": "Hear why GPT-5.5’s dual focus on capability and security is significant—and debate    │
│  whether the rapid scaling and commercial barriers are helping or hurting widespread AI adoption.",             │
│        "source": "https://techcrunch.com/2026/04/23/openai-chatgpt-gpt-5-5-ai-model-superapp/"                  │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "GPT-5.5 and the Cyber Frontier: Trusted Access and Specialized LLMs",                          │
│        "summary": "OpenAI’s specialized GPT-5.5-Cyber model, announced alongside GPT-5.5, targets security and  │
│  compliance in AI-driven cyber environments. This marks a major push in using LLMs for real-time threat         │
│  response, autonomous security operations, and trusted access controls, signalling a new era where language     │
│  models play critical roles in cybersecurity.",                                                                 │
│        "listener_angle": "Explore the shifting landscape as leading LLMs become specialized, not just           │
│  smarter—what does this mean for industry, privacy, and risk mitigation?",                                      │
│        "source": "https://openai.com/sl-SI/index/gpt-5-5-with-trusted-access-for-cyber/"                        │
│      },                                                                                                         │
│      {                                                 

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 8fde34d3-3416-410a-b9dc-34bafde30432                                                                     │
│  Agent: AI Podcast Researcher                                                                                   │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: a5e5daaf-2148-4803-9300-9c49a72b7024                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: {                                                                                                │
│    "topic": "AI LLMs: The 2026 Spring Surge",                                                                   │
│    "episode_angle": "Spring 2026 has brought a wave of innovation in Large Language Models (LLMs), with new     │
│  releases pushing boundaries in performance, accessibility, specialization, and governance. We explore          │
│  OpenAI’s GPT-5.5 and its cyber-focused sibling, Baidu’s ERNIE 5.1, Google’s Gemma 4, and the open-source       │
│  momentum from MiniMax—plus the implications of rapid, ongoing LLM progress.",                                  │
│    "items": [                                                                                                   │
│      {                                                                                                          │
│        "title": "OpenAI’s GPT-5.5: A Leap Toward the 'AI Super App'",                                           │
│        "summary": "OpenAI’s GPT-5.5, unveiled in April 2026, has been hailed as introducing 'a new class of     │
│  intelligence.' This LLM vastly improves reasoning, multi-modal abilities, and integration of specialized       │
│  capabilities (notably through GPT-5.5-Cyber for trusted cyber operations). OpenAI claims its model             │
│  outperforms previous versions on numerous benchmarks and is shaping a future where LLMs underpin               │
│  multi-functional AI 'super apps.' However, the API’s doubled price has sparked debate on access and            │
│  democratization.",                                                                                             │
│        "listener_angle": "Hear why GPT-5.5’s dual focus on capability and security is significant—and debate    │
│  whether the rapid scaling and commercial barriers are helping or hurting widespread AI adoption.",             │
│        "source": "https://techcrunch.com/2026/04/23/openai-chatgpt-gpt-5-5-ai-model-superapp/"                  │
│      },                                                                                                         │
│      {                                                                                                          │
│        "title": "GPT-5.5 and the Cyber Frontier: Trusted Access and Specialized LLMs",                          │
│        "summary": "OpenAI’s specialized GPT-5.5-Cyber model, announced alongside GPT-5.5, targets security and  │
│  compliance in AI-driven cyber environments. This marks a major push in using LLMs for real-time threat         │
│  response, autonomous security operations, and trusted access controls, signalling a new era where language     │
│  models play critical roles in cybersecurity.",                                                                 │
│        "listener_angle": "Explore the shifting landscape as leading LLMs become specialized, not just           │
│  smarter—what does this mean for industry, privacy, and risk mitigation?",                                      │
│        "source": "https://openai.com/sl-SI/index/gpt-5-5-with-trusted-access-for-cyber/"                        │
│      },                                               

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: ced25b69-a427-4f11-a9c4-48a608d10335                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/ced25b69-a427-4f11-a9c4-48a608d10335?access_code=TRA │
│ CE-11a60ccaf0                                                                                                   │
│ 🔑 Access Code: TRACE-11a60ccaf0                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [6]:
report = structured_result.pydantic
print(f"Episode angle: {report.episode_angle}")
print(f"Findings: {len(report.items)}")
print(f"Host questions: {len(report.host_questions)}")
for item in report.items[:3]:
    print(f"  - {item.title}")


Episode angle: Spring 2026 has brought a wave of innovation in Large Language Models (LLMs), with new releases pushing boundaries in performance, accessibility, specialization, and governance. We explore OpenAI’s GPT-5.5 and its cyber-focused sibling, Baidu’s ERNIE 5.1, Google’s Gemma 4, and the open-source momentum from MiniMax—plus the implications of rapid, ongoing LLM progress.
Findings: 6
Host questions: 8
  - OpenAI’s GPT-5.5: A Leap Toward the 'AI Super App'
  - GPT-5.5 and the Cyber Frontier: Trusted Access and Specialized LLMs
  - ERNIE 5.1: Baidu’s Next-Gen Model Sets New Leaderboard Records


---

## 2. Code Guardrail

Structured output gives us typed fields. A **guardrail** lets us programmatically reject weak outputs and force the agent to retry.


In [7]:
def validate_research_report(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if len(report.items) < 5:
        return (False, "The research report must include at least 5 findings.")
    if len(report.host_questions) < 3:
        return (False, "The report must include at least 3 host questions.")
    return (True, report)


research_task = Task(
    description=(
        "Conduct podcast research about {topic}. Find recent releases, articles, topics, "
        "and progress. Use the current date context: {current_month} {current_year}."
    ),
    expected_output="A structured research report with at least 5 findings and 3 host questions.",
    output_pydantic=ResearchReport,
    guardrail=validate_research_report,
    agent=researcher,
)


If the agent returns fewer than 5 findings or fewer than 3 questions, CrewAI rejects the output and asks it to try again — automatically.

---

## 3. No-Code Guardrail

For simpler validation, you can skip the Python function entirely and write the guardrail in plain English. CrewAI uses the LLM itself to judge whether the output passes.


In [8]:
reporting_analyst = Agent(
    role="AI Podcast Reporting Analyst",
    goal="Create detailed reports based on {topic} research findings",
    backstory=(
        "You are a meticulous analyst with a keen eye for detail. "
        "You turn complex research into clear podcast briefing documents."
    ),
    llm=llm,
    verbose=True,
    max_iter=4,
    max_rpm=10,
    max_execution_time=240,
    respect_context_window=True,
)

reporting_task = Task(
    description=(
        "Review the research report and organize each topic into a full section. "
        "Make the report detailed, useful for podcast hosts, and easy to turn into a script."
    ),
    expected_output=(
        "A markdown report with episode angle, 3 or more sections, key facts, "
        "listener takeaways, and suggested transitions between topics."
    ),
    guardrail=(
        "The report must include at least 3 clearly labeled sections with headers. "
        "Each section must contain at least 2 bullet points of useful facts."
    ),
    agent=reporting_analyst,
    context=[research_task],
)


---

## 4. Add the Scriptwriter

The scriptwriter turns the structured report into a two-host podcast dialogue. This is where the workflow starts feeling like a production system.


In [9]:
scriptwriter = Agent(
    role="Podcast Scriptwriter",
    goal="Create a fun, natural, interesting podcast script from a research report on {topic}",
    backstory=(
        "You are a naturally talented scriptwriter who turns technical reports into engaging, "
        "funny, and articulate podcast conversations between two hosts. You know where to insert "
        "jokes, laughs, and where and how to bring up technical details."
    ),
    llm=llm,
    verbose=True,
    max_iter=4,
    max_rpm=8,
    max_execution_time=300,
    respect_context_window=True,
)

# Prompt Engineering: Chain-of-Thought — sequential reasoning steps guide the agent before it writes
scripting_task = Task(
    description=(
        "Review the podcast report and write a script between two knowledgeable hosts. "
        "First, identify the 3-4 most compelling moments or surprising reveals in the report — these become the episode peaks. "
        "Then plan a narrative arc: cold open hook, topic order, and a closing takeaway. "
        "Finally, write the full dialogue between Joe and Jane following that arc. "
        "The only speakers are Joe and Jane. Keep it lively, natural, and under 100 exchanges. "
        "Include speaker labels, sound effects cues, laughs, and topic transitions."
    ),
    expected_output=(
        "A complete podcast script with speaker labels for Joe and Jane, a cold open, "
        "topic transitions, concise technical explanations, and a closing summary."
    ),
    guardrail=(
        "The script must use only Joe and Jane as speaker labels. "
        "It must include at least 12 lines of dialogue."
    ),
    agent=scriptwriter,
    context=[reporting_task],
)


---

## 5. Full 3-Agent Pipeline

```text
Researcher ──context──> Reporting Analyst ──context──> Scriptwriter
```


In [10]:
podcaster_crew = Crew(
    agents=[researcher, reporting_analyst, scriptwriter],
    tasks=[research_task, reporting_task, scripting_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=30,
)

inputs = {
    "topic": "AI LLMs",
    "current_month": datetime.now().strftime("%B"),
    "current_year": str(datetime.now().year),
}

final_result = podcaster_crew.kickoff(inputs=inputs)
final_result


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 16a4d4ad-8b4c-4aeb-ad8c-39a3e711be14                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Task: Conduct podcast research about AI LLMs. Find recent releases, articles, topics, and progress. Use the    │
│  current date context: May 2026.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Podcast Researcher                                                                                   │
│                                                                                                                 │
│  Thought: Thought: I should search for the most recent (2026) releases, news, articles, and progress on AI      │
│  LLMs to provide detailed, structured research findings related to current events and advancements.             │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"latest AI LLM releases and research breakthroughs 2026\", \"start_published_date\":      │
│  \"2026-01-01\", \"end_published_date\": \"2026-05-15\", \"include_domains\": null}"                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 🛡️ Guardrail Success ──────────────────────────────────────────────╮
│                                                                                                                 │
│  Guardrail Passed                                                                                               │
│  Name: Validation Successful                                                                                    │
│  Status: ✅ Validated                                                                                           │
│  Attempts: 1                                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 78c8a16a-fc9e-446c-baa9-0a4048bceb76                                                                     │
│  Agent: Podcast Scriptwriter                                                                                    │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 16a4d4ad-8b4c-4aeb-ad8c-39a3e711be14                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: [Sound cue: Upbeat electronic intro music fades in, then lowers under voices]                    │
│                                                                                                                 │
│  Joe: (excited, teasing) All right Jane—pop quiz! Which new AI model cooks your dinner, writes your thesis,     │
│  books your travel, and gives you a pep talk when your sourdough fails?                                         │
│                                                                                                                 │
│  Jane: (laughs) I mean, it’s gotta be the one with the “.5” in its name, right? Joe, are you still upset it     │
│  can’t fold your laundry… yet?                                                                                  │
│                                                                                                                 │
│  Joe: Only mildly. (dramatic sigh) Welcome back to “AI Unplugged,” where this week we are deep diving into      │
│  language models that are, frankly, starting to sound more like digital sidekicks—and less like chatbots.       │
│                                                                                                                 │
│  Jane: Today on the show: We’re talking about four truly wild peaks—from the all-purpose “super app” GPT-5.5,   │
│  to China’s mind-boggling ERNIE 5.1, through the creative wonders of Muse Spark, and finally, the turbocharged  │
│  under-the-hood upgrades from NVIDIA’s Star Elastic and a tiny company you’ll want to remember: SubQ.           │
│                                                                                                                 │
│  Joe: By the end, get ready to ask not just what AI can do, but how it’s going to change the way you work,      │
│  create—and, we hope, not destroy dinner.                                                                       │
│                                                                                                                 │
│  [Intro music ramps down, SFX: “ding!”]                                                                         │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  Joe: Okay, Jane. First stop: the “Super App” era. OpenAI dropped GPT-5.5 in April—it’s being hailed as the     │
│  ‘Swiss Army Knife’ of LLMs.                                                                                    │
│                                                                                                                 │
│  Jane: (snickers) More like the Batman utility belt! Forget just chatting—GPT-5.5 is context-aware, integrates  │
│  live web search, crunches data, automates all sorts of routine tasks, and even talks back in that charming,    │
│  ever-so-calming digital voice.                                                                                 │
│                                                       

╭─────────────────────────────────────────── Trace Batch Finalization ────────────────────────────────────────────╮
│ ✅ Trace batch finalized with session ID: f471247c-02bc-48ab-93a1-2e2597453a72                                  │
│                                                                                                                 │
│ 🔗 View here:                                                                                                   │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/f471247c-02bc-48ab-93a1-2e2597453a72?access_code=TRA │
│ CE-b6557b4e31                                                                                                   │
│ 🔑 Access Code: TRACE-b6557b4e31                                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw="[Sound cue: Upbeat electronic intro music fades in, then lowers under voices]\n\nJoe: (excited, teasing) All right Jane—pop quiz! Which new AI model cooks your dinner, writes your thesis, books your travel, and gives you a pep talk when your sourdough fails?\n\nJane: (laughs) I mean, it’s gotta be the one with the “.5” in its name, right? Joe, are you still upset it can’t fold your laundry… yet?\n\nJoe: Only mildly. (dramatic sigh) Welcome back to “AI Unplugged,” where this week we are deep diving into language models that are, frankly, starting to sound more like digital sidekicks—and less like chatbots.\n\nJane: Today on the show: We’re talking about four truly wild peaks—from the all-purpose “super app” GPT-5.5, to China’s mind-boggling ERNIE 5.1, through the creative wonders of Muse Spark, and finally, the turbocharged under-the-hood upgrades from NVIDIA’s Star Elastic and a tiny company you’ll want to remember: SubQ.\n\nJoe: By the end, get ready to ask not just wh

---

## Recap

| Step | The Problem | The Fix | CrewAI API |
|------|------------|---------|------------|
| 1 | Raw text is fragile | Typed Pydantic models | `output_pydantic=Model` |
| 2 | Weak output passes unchecked | Code guardrail | `guardrail=validate_fn` |
| 3 | Writing validators is tedious | Plain-English guardrail | `guardrail="..."` |
| 4 | No script output | Scriptwriter agent | `Agent(...)` + `context=[reporting_task]` |
| 5 | Full production flow | 3-agent crew | `Crew(agents=[...], tasks=[...])` |

**Next:** Open `03_voice_generation.ipynb` to build custom tools and generate the final podcast audio.
